# 第6章：Fine-tuning for Classification

**目标：** 将预训练好的 GPT 模型改造为**垃圾短信分类器**，学习分类微调的完整流程

```
微调类型概览 → 准备数据集 → 创建 DataLoader → 加载预训练权重 → 添加分类头 → 训练 → 评估 → 使用
```

**前置回顾（第2-5章）：**
- 第2章：文本 → Token IDs → DataLoader
- 第3章：Causal Multi-Head Attention
- 第4章：完整 GPT 模型架构
- 第5章：预训练 + 加载 GPT-2 权重 + 文本生成
- 现在的问题：预训练模型只会「续写」文本，**不能做分类** → **需要微调！**

---

## 6.1 微调的类型 ⭐

LLM 最常见的两种微调方式：

| 类型 | 目标 | 输出 | 代表应用 |
|------|------|------|----------|
| **分类微调（Classification）** | 让模型输出固定的类别标签 | `spam` / `not spam` | 情感分析、垃圾邮件检测 |
| **指令微调（Instruction）** | 让模型按指令生成文本 | 自由文本 | ChatGPT 式对话 |

**本章聚焦分类微调：**
- 分类微调 = 给模型加一个「分类头」，输出**固定数量的类别**
- 模型只能预测训练过程中见过的类别（例如 spam / not spam）
- 相比指令微调，分类模型更加**专注**，通常也更容易训练

> 💡 **关键洞察：** 分类微调类似于用 CNN 做手写数字分类——都是替换输出层，冻结骨干网络，然后用有标签数据训练。

指令微调将在**第7章**详细讨论。

---

## 6.2 准备数据集 ⭐⭐

我们使用 **SMS Spam Collection** 数据集——包含 5,572 条短信，标注为 `ham`（正常）或 `spam`（垃圾）。

**数据处理流程：**
1. 下载并解压数据
2. 加载为 pandas DataFrame
3. 平衡类别（欠采样多数类）
4. 转换标签：`ham → 0`，`spam → 1`
5. 划分训练/验证/测试集（70%/10%/20%）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import os

# ─── 第4章的所有组件（直接复用）───────────────────────────────────

# GPT-2 Small (124M) 的超参数配置
GPT_CONFIG_124M = {
    "vocab_size": 50257,      # BPE 词汇表大小
    "context_length": 1024,   # 最大序列长度（位置编码上限）
    "emb_dim": 768,           # 嵌入维度 / 隐藏层维度
    "n_heads": 12,            # 多头注意力的头数（每头 768/12=64 维）
    "n_layers": 12,           # Transformer Block 堆叠层数
    "drop_rate": 0.1,         # Dropout 概率
    "qkv_bias": False,        # Q/K/V 线性投影是否带偏置项
}


class MultiHeadAttention(nn.Module):
    """多头因果自注意力（第3章）"""
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out 必须能被 num_heads 整除"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout  = nn.Dropout(dropout)

        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(
            self.mask[:num_tokens, :num_tokens].bool(), -torch.inf
        )
        attn_weights = F.softmax(attn_scores / self.head_dim**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vecs = attn_weights @ values
        context_vecs = context_vecs.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        context_vecs = self.out_proj(context_vecs)
        return context_vecs


class LayerNorm(nn.Module):
    """层归一化（第4章）"""
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    """GELU 激活函数（第4章）"""
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    """前馈网络（第4章）"""
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    """Transformer Block（第4章）"""
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class GPTModel(nn.Module):
    """完整 GPT 模型（第4章）"""
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


def generate_text_simple(model, idx, max_new_tokens, context_size):
    """贪心解码文本生成（第4章）"""
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx


# ─── 第5章的权重加载工具 ─────────────────────────────────────────

def assign(left, right):
    """将 numpy 权重赋值给 PyTorch 参数"""
    if left.shape != right.shape:
        raise ValueError(f"Shape 不匹配: {left.shape} vs {right.shape}")
    return nn.Parameter(torch.tensor(right))


def load_weights_into_gpt(gpt, params):
    """将 Hugging Face GPT-2 权重加载到我们的 GPTModel 中（第5章）"""
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte.weight"])
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe.weight"])

    for b in range(len(gpt.trf_blocks)):
        q_w, k_w, v_w = np.split(
            params[f"h.{b}.attn.c_attn.weight"], 3, axis=-1
        )
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T
        )
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T
        )
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T
        )

        q_b, k_b, v_b = np.split(
            params[f"h.{b}.attn.c_attn.bias"], 3, axis=-1
        )
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b
        )
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b
        )
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b
        )

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params[f"h.{b}.attn.c_proj.weight"].T
        )
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params[f"h.{b}.attn.c_proj.bias"]
        )

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params[f"h.{b}.mlp.c_fc.weight"].T
        )
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params[f"h.{b}.mlp.c_fc.bias"]
        )
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params[f"h.{b}.mlp.c_proj.weight"].T
        )
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params[f"h.{b}.mlp.c_proj.bias"]
        )

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params[f"h.{b}.ln_1.weight"]
        )
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params[f"h.{b}.ln_1.bias"]
        )
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params[f"h.{b}.ln_2.weight"]
        )
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params[f"h.{b}.ln_2.bias"]
        )

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["ln_f.weight"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["ln_f.bias"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte.weight"])


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    return torch.tensor(encoded).unsqueeze(0)


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())


# ─── 初始化 ────────────────────────────────────────────────────
tokenizer = tiktoken.get_encoding("gpt2")
print("第4-5章组件加载完毕 ✓")

### 下载并加载 SMS Spam 数据集

In [ ]:
import urllib.request
import zipfile
from pathlib import Path
import pandas as pd

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} 已存在，跳过下载。")
        return

    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"文件已下载并保存为 {data_file_path}")


download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

In [ ]:
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
print(f"→ 数据集大小: {len(df)} 条短信")
print(f"→ 类别分布:\n{df['Label'].value_counts()}")
df.head()

### 平衡数据集

数据集中 ham（正常）远多于 spam（垃圾），我们通过**欠采样（undersampling）** 使两类数量相等。

In [ ]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])
    return balanced_df


balanced_df = create_balanced_dataset(df)
print(f"→ 平衡后类别分布:\n{balanced_df['Label'].value_counts()}")

In [ ]:
# 将字符串标签转为整数：ham → 0, spam → 1
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})
balanced_df.head()

### 划分训练/验证/测试集

In [ ]:
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]
    return train_df, validation_df, test_df


train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

print(f"→ 训练集: {len(train_df)} | 验证集: {len(validation_df)} | 测试集: {len(test_df)}")

### ✏️ 练习

1. 查看 `balanced_df` 中最长和最短的短信各有多少个词？多少个 token？
2. 如果不做欠采样，直接用不平衡的数据训练，模型会出现什么问题？
3. **进阶**：除了欠采样，还有哪些处理类别不平衡的方法？试着搜索 `imbalanced-learn` 库。

In [ ]:
# 在这里做实验

---

## 6.3 创建 DataLoader ⭐⭐

不同短信长度不一，需要**填充（padding）**到统一长度才能组成 batch。

**策略：**
- 用 `<|endoftext|>`（token ID = 50256）作为 padding token
- 所有序列填充到训练集中最长的序列长度
- 超过最大长度的序列会被截断

In [ ]:
from torch.utils.data import Dataset, DataLoader


class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        # 预先分词
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            # 截断超长序列
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # 填充到统一长度
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        return max(len(encoded_text) for encoded_text in self.encoded_texts)

In [ ]:
# 创建三个数据集
train_dataset = SpamDataset(csv_file="train.csv", max_length=None, tokenizer=tokenizer)
print(f"→ 训练集最长序列: {train_dataset.max_length} tokens")

val_dataset = SpamDataset(
    csv_file="validation.csv", max_length=train_dataset.max_length, tokenizer=tokenizer
)
test_dataset = SpamDataset(
    csv_file="test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer
)

In [ ]:
# 创建 DataLoader
num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset, batch_size=batch_size,
    shuffle=True, num_workers=num_workers, drop_last=True,
)
val_loader = DataLoader(
    dataset=val_dataset, batch_size=batch_size,
    num_workers=num_workers, drop_last=False,
)
test_loader = DataLoader(
    dataset=test_dataset, batch_size=batch_size,
    num_workers=num_workers, drop_last=False,
)

# 验证 batch 维度
for input_batch, target_batch in train_loader:
    pass
print(f"→ 输入 batch: {input_batch.shape}")
print(f"→ 标签 batch: {target_batch.shape}")
print(f"→ {len(train_loader)} 训练 batches | {len(val_loader)} 验证 batches | {len(test_loader)} 测试 batches")

### ✏️ 练习

1. 改变 `batch_size`（如 4、16、32），观察 batch 数量的变化。更大的 batch_size 有什么利弊？
2. 如果不做 padding，直接用变长序列训练，会遇到什么问题？
3. **进阶**：当前我们把所有序列 pad 到数据集最长长度。能否改为 pad 到**每个 batch 的最长长度**？这样做有什么好处？

In [ ]:
# 在这里做实验

---

## 6.4 加载预训练权重 ⭐⭐

我们复用第5章的做法，从 Hugging Face 下载 GPT-2 (124M) 的预训练权重并加载到模型中。

**关键配置变化：**
- `drop_rate` 设为 **0.0**（微调时不需要太多 dropout）
- `qkv_bias` 设为 **True**（GPT-2 原版带偏置）

In [ ]:
from transformers import GPT2Model as HF_GPT2Model

def download_and_load_gpt2(model_size, models_dir):
    """从 Hugging Face 下载 GPT-2 权重"""
    model_hf = HF_GPT2Model.from_pretrained(model_size, cache_dir=models_dir)
    model_hf.eval()
    params = {}
    for name, param in model_hf.named_parameters():
        params[name] = param.detach().numpy()
    for name, buf in model_hf.named_buffers():
        params[name] = buf.detach().numpy()
    return params


CHOOSE_MODEL = "gpt2"

BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,        # 微调时 dropout 设为 0
    "qkv_bias": True,        # GPT-2 原版带偏置
    "emb_dim": 768,
    "n_layers": 12,
    "n_heads": 12,
}

assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"数据集长度 {train_dataset.max_length} 超过模型上下文长度 {BASE_CONFIG['context_length']}"
)

In [ ]:
# 下载并加载权重
models_dir = os.path.join("..", "models", CHOOSE_MODEL)
os.makedirs(models_dir, exist_ok=True)

print(f"正在下载 {CHOOSE_MODEL} 模型...")
params = download_and_load_gpt2(CHOOSE_MODEL, models_dir)
print(f"下载完成！参数数量: {len(params)}")

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()
print("\n预训练权重加载完毕 ✓")

### 验证权重：让模型生成文本

In [ ]:
# 验证模型可以生成连贯文本
text = "Every effort moves you"
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)
print(f"→ 输入: {text}")
print(f"→ 生成: {token_ids_to_text(token_ids, tokenizer)}")

### 试试用 prompt 做分类？

在微调之前，先看看预训练模型能否直接通过 prompt 来判断垃圾短信：

In [ ]:
text = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text, tokenizer),
    max_new_tokens=23,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))

> 💡 **观察：** 预训练模型并不擅长「遵循指令」——它只会续写文本，不会回答 yes/no 问题。
> 这正是我们需要**微调**的原因！指令微调会在第7章介绍。

---

## 6.5 添加分类头 ⭐⭐⭐

这是本章的**核心步骤**：

1. **冻结所有参数** — 让预训练权重不被破坏
2. **替换 `out_head`** — 从 50,257 维（词汇表）→ 2 维（spam / not spam）
3. **解冻最后一层 Transformer Block + final_norm** — 适度微调提升效果

```
为什么用最后一个 token 做分类？

在 Causal Attention 中，每个 token 只能看到自己和之前的 token。
因此最后一个 token 拥有整个序列的信息 → 最适合做分类判断。
```

In [ ]:
# 第一步：冻结所有参数
for param in model.parameters():
    param.requires_grad = False

# 第二步：替换输出层 (50,257 → 2)
torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(
    in_features=BASE_CONFIG["emb_dim"],
    out_features=num_classes
)

# 第三步：解冻最后一层 Transformer Block 和 final_norm
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in model.final_norm.parameters():
    param.requires_grad = True

# 统计可训练参数
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"→ 总参数: {total_params:,}")
print(f"→ 可训练参数: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

### 验证新的输出维度

In [ ]:
inputs = tokenizer.encode("Do you have time")
inputs = torch.tensor(inputs).unsqueeze(0)
print(f"→ 输入: {inputs}, shape: {inputs.shape}")

with torch.no_grad():
    outputs = model(inputs)
print(f"→ 输出 shape: {outputs.shape}  (batch=1, seq=4, classes=2)")

# 取最后一个 token 的输出作为分类依据
print(f"→ 最后一个 token 的 logits: {outputs[:, -1, :]}")

### ✏️ 练习

1. 如果只训练 `out_head`（不解冻最后一层 TransformerBlock），准确率会下降多少？
2. 如果解冻**所有**层（full fine-tuning），训练速度和效果会怎样？
3. **进阶**：搜索 LoRA（Low-Rank Adaptation），了解另一种参数高效微调方法。

In [ ]:
# 在这里做实验

---

## 6.6 计算分类损失和准确率 ⭐⭐

分类微调的评估指标：
- **交叉熵损失（Cross-Entropy Loss）** — 可微分，用于训练
- **分类准确率（Accuracy）** — 直观，用于评估

与第5章预训练的区别：
- 预训练时，我们优化**所有位置**的 next-token prediction loss
- 分类微调时，我们只取**最后一个 token** 的输出做分类

$$\text{Prediction} = \arg\max(\text{model}(x)[:, -1, :])$$

In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    """计算整个 data_loader 上的分类准确率"""
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)
            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # 最后一个 token
            predicted_labels = torch.argmax(logits, dim=-1)
            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples


def calc_loss_batch(input_batch, target_batch, model, device):
    """计算单个 batch 的交叉熵损失（只看最后一个 token）"""
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, -1, :]
    loss = F.cross_entropy(logits, target_batch)
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    """计算整个 data_loader 上的平均损失"""
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [ ]:
# 选择设备
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"→ 使用设备: {device}")
model.to(device)

# 训练前的基线指标
torch.manual_seed(123)
with torch.no_grad():
    train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
    val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print(f"\n训练前基线指标:")
print(f"  训练准确率: {train_accuracy*100:.2f}%")
print(f"  验证准确率: {val_accuracy*100:.2f}%")
print(f"  训练损失: {train_loss:.3f}")
print(f"  验证损失: {val_loss:.3f}")

> 💡 **观察：** 训练前准确率大约在 50% 左右——相当于随机猜测。这很正常，因为分类头是随机初始化的。

---

## 6.7 训练分类器 ⭐⭐⭐

训练循环与第5章的 `train_model_simple` 基本相同，主要差异：
1. 跟踪 `examples_seen`（而非 `tokens_seen`）
2. 每个 epoch 后计算准确率（而非生成文本样本）

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def train_classifier_simple(model, train_loader, val_loader, optimizer, device,
                            num_epochs, eval_freq, eval_iter):
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    examples_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            examples_seen += input_batch.shape[0]
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # 每个 epoch 结束后计算准确率
        train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=eval_iter)
        val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=eval_iter)
        print(f"Training accuracy: {train_accuracy*100:.2f}% | ", end="")
        print(f"Validation accuracy: {val_accuracy*100:.2f}%")
        train_accs.append(train_accuracy)
        val_accs.append(val_accuracy)

    return train_losses, val_losses, train_accs, val_accs, examples_seen

In [ ]:
import time

# 🚀 开始训练！
start_time = time.time()

torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
num_epochs = 5

train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
print(f"\n训练完成！耗时 {(end_time - start_time) / 60:.2f} 分钟。")

### 可视化训练过程

In [ ]:
def plot_values(epochs_seen, examples_seen, train_values, val_values, label="loss"):
    fig, ax1 = plt.subplots(figsize=(6, 3.5))
    ax1.plot(epochs_seen, train_values, label=f"训练 {label}")
    ax1.plot(epochs_seen, val_values, linestyle="-.", label=f"验证 {label}")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel(label.capitalize())
    ax1.legend()

    ax2 = ax1.twiny()
    ax2.plot(examples_seen, train_values, alpha=0)
    ax2.set_xlabel("样本数")

    fig.tight_layout()
    plt.show()


# 绘制损失曲线
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))
plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses, label="loss")

In [ ]:
# 绘制准确率曲线
epochs_tensor = torch.linspace(0, num_epochs, len(train_accs))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_accs))
plot_values(epochs_tensor, examples_seen_tensor, train_accs, val_accs, label="accuracy")

### 最终评估

In [ ]:
# 在完整数据集上计算最终准确率
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"训练准确率: {train_accuracy*100:.2f}%")
print(f"验证准确率: {val_accuracy*100:.2f}%")
print(f"测试准确率: {test_accuracy*100:.2f}%")

> 💡 **观察：** 测试集准确率略低于训练/验证集，这说明模型存在轻微的过拟合。可以通过增大 `drop_rate` 或 `weight_decay` 来缓解。

### ✏️ 练习

1. 调整学习率（如 1e-4, 1e-5），观察训练曲线和最终准确率的变化。
2. 增加 `num_epochs` 到 10，观察是否出现过拟合（训练 loss 下降但验证 loss 上升）。
3. **进阶**：添加梯度裁剪（`torch.nn.utils.clip_grad_norm_`），看看对训练稳定性有什么影响。

In [ ]:
# 在这里做实验

---

## 6.8 使用分类器 ⭐

训练完成后，我们可以用模型对新短信进行分类：

In [ ]:
def classify_review(text, model, tokenizer, device, max_length=None, pad_token_id=50256):
    """对一条文本进行 spam/not spam 分类"""
    model.eval()

    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]

    input_ids = input_ids[:min(max_length, supported_context_length)]
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0)

    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]
    predicted_label = torch.argmax(logits, dim=-1).item()

    return "spam" if predicted_label == 1 else "not spam"

In [ ]:
# 测试：垃圾短信
text_1 = (
    "You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award."
)
print(f"\"...{text_1[:50]}...\" → {classify_review(text_1, model, tokenizer, device, max_length=train_dataset.max_length)}")

# 测试：正常短信
text_2 = (
    "Hey, just wanted to check if we're still on"
    " for dinner tonight? Let me know!"
)
print(f"\"...{text_2[:50]}...\" → {classify_review(text_2, model, tokenizer, device, max_length=train_dataset.max_length)}")

### 保存和加载模型

In [ ]:
# 保存微调后的模型
torch.save(model.state_dict(), "spam_classifier.pth")
print("模型已保存为 spam_classifier.pth")

# 加载方式示例（在新的 session 中使用）：
# model = GPTModel(BASE_CONFIG)
# model.out_head = torch.nn.Linear(in_features=768, out_features=2)
# model.load_state_dict(torch.load("spam_classifier.pth", map_location=device, weights_only=True))
# model.eval()

### ✏️ 练习

1. 自己编写几条短信（中文或英文），测试分类器的效果。
2. 模型对哪类短信容易判断错误？分析原因。
3. **进阶**：将保存的模型在新的 notebook 中加载并使用。

In [ ]:
# 在这里做实验

---

## 6.9 完整流程回顾

In [ ]:
print("""
═══════════════════════════════════════════════════════════
          第 6 章：分类微调 — 完整流程
═══════════════════════════════════════════════════════════

  1. 了解微调类型（分类 vs 指令）
     ↓
  2. 下载并准备 SMS Spam 数据集
     ↓  平衡类别 → 划分 train/val/test
  3. 创建 SpamDataset + DataLoader
     ↓  tokenize → padding → batch
  4. 加载预训练 GPT-2 权重
     ↓  验证文本生成能力
  5. 添加分类头
     ↓  冻结 backbone → 替换 out_head → 解冻最后一层
  6. 计算分类损失和准确率
     ↓  cross_entropy(logits[:, -1, :], labels)
  7. 训练分类器
     ↓  5 epochs, AdamW, lr=5e-5
  8. 使用分类器对新文本做预测
     ↓  tokenize → pad → forward → argmax
  9. 第 7 章：指令微调 — 让模型学会对话！

═══════════════════════════════════════════════════════════

关键收获:
  • 分类微调 = 冻结 backbone + 替换输出头 + 用有标签数据训练
  • 只取最后一个 token 的输出做分类（因为 causal attention）
  • 解冻最后几层可以显著提升效果
  • 只需训练约 7% 的参数就能达到 ~95% 以上的准确率
""")

---

## 📝 本章核心 Checklist

学完本章后，检查你是否能回答以下问题：

- [ ] 分类微调和指令微调有什么区别？各适用于什么场景？
- [ ] 为什么要用最后一个 token 的输出做分类？
- [ ] 为什么要冻结大部分参数？只训练哪些层？
- [ ] `out_head` 替换前后，模型的输出维度有什么变化？
- [ ] `SpamDataset` 中 padding 的作用是什么？为什么用 `<|endoftext|>` 做 padding token？
- [ ] 训练中的 `calc_loss_batch` 与第5章有什么区别？
- [ ] 怎样判断模型是否过拟合？有哪些缓解方法？
- [ ] 如何保存和加载微调后的模型？

全部能回答 → 进入**第 7 章：指令微调**——让模型学会像 ChatGPT 一样对话！